# Reporte de Validación y Riesgos: Proyecto SOLARIS

## Módulo 4: QUANTUM ANALYTICS

### Validación robusta, distributional shift, fairness y marco ético

Paso 1. Separa tus datos por ventana temporal

In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "solaris_datos_limpios.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dimensiones del dataset:", df.shape)
print(df.columns.tolist())

Dimensiones del dataset: (15000, 11)
['id_cliente', 'edad', 'ingreso_mensual', 'nivel_educativo', 'ciudad', 'fecha_vinculacion', 'saldo_actual', 'cuotas_pagadas', 'cuotas_totales', 'estado_cuenta', 'score_externo']


In [33]:
df["fecha_vinculacion"] = pd.to_datetime(
    df["fecha_vinculacion"],
    errors="coerce"
)

df_modelo = df[
    df["estado_cuenta"].isin(["Al día", "Moroso"])
].copy()

print("Fecha mínima:", df_modelo["fecha_vinculacion"].min())
print("Fecha máxima:", df_modelo["fecha_vinculacion"].max())
print("Registros para análisis:", len(df_modelo))
print("Fechas nulas:", df_modelo["fecha_vinculacion"].isna().sum())

Fecha mínima: 2015-01-01 00:00:00
Fecha máxima: 2025-06-30 00:00:00
Registros para análisis: 12300
Fechas nulas: 0


In [34]:
fecha_corte = (
    df_modelo["fecha_vinculacion"].max()
    - pd.Timedelta(weeks=6)
)

df_referencia = df_modelo[
    df_modelo["fecha_vinculacion"] <= fecha_corte
].copy()

df_reciente = df_modelo[
    df_modelo["fecha_vinculacion"] > fecha_corte
].copy()

print("Fecha de corte:", fecha_corte)

print("\nTamaño referencia:", len(df_referencia))
print("Tamaño reciente:", len(df_reciente))

print(
    "\nPeriodo referencia:",
    df_referencia["fecha_vinculacion"].min(),
    "a",
    df_referencia["fecha_vinculacion"].max()
)

print(
    "Periodo reciente:",
    df_reciente["fecha_vinculacion"].min(),
    "a",
    df_reciente["fecha_vinculacion"].max()
)

Fecha de corte: 2025-05-19 00:00:00

Tamaño referencia: 12164
Tamaño reciente: 136

Periodo referencia: 2015-01-01 00:00:00 a 2025-05-19 00:00:00
Periodo reciente: 2025-05-20 00:00:00 a 2025-06-30 00:00:00


## Datos analizados

Se analizaron únicamente los distribuidores pertenecientes a las clases utilizadas por el modelo (`Al día` y `Moroso`), excluyendo los registros con estado `Cancelado`.

- **Datos de referencia:** 12.164 observaciones, correspondientes a distribuidores vinculados entre el **1 de enero de 2015** y el **19 de mayo de 2025**.
- **Datos recientes:** 136 observaciones, correspondientes a distribuidores vinculados entre el **20 de mayo de 2025** y el **30 de junio de 2025**.
- **Fecha de corte:** 19 de mayo de 2025.
- **Ventana reciente:** 6 semanas.
- **Total analizado:** 12.300 observaciones.

La ventana reciente es considerablemente menor que la muestra histórica, por lo que los resultados de las pruebas de cambio de distribución se interpretarán como evidencia de monitoreo y no como prueba definitiva de degradación.

Adicionalmente, `fecha_vinculacion` representa la fecha de incorporación del distribuidor y no una fecha de transacción o de scoring. Por esta razón, el análisis permite evaluar cambios en el perfil de los distribuidores vinculados recientemente frente al histórico, pero no constituye por sí solo evidencia de cambio en el comportamiento de clientes ya existentes.

Paso 2. Identifica las variables más importantes del modelo

In [35]:
import joblib

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "artifacts"
    / "solaris_rf_v1_20260916.joblib"
)

modelo = joblib.load(MODEL_PATH)

preprocessor = modelo.named_steps["preprocessor"]
modelo_rf = modelo.named_steps["classifier"]

print("Modelo:", type(modelo_rf))

Modelo: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [36]:
feature_names_transformadas = preprocessor.get_feature_names_out()

importancias_transformadas = pd.Series(
    modelo_rf.feature_importances_,
    index=feature_names_transformadas
).sort_values(ascending=False)

print("Número de variables transformadas:", len(importancias_transformadas))

display(importancias_transformadas.head(15))

Número de variables transformadas: 41


num__saldo_actual                     0.355649
num__cuotas_pagadas                   0.239167
num__score_externo                    0.161697
num__cuotas_totales                   0.078277
num__edad                             0.044727
num__ingreso_mensual                  0.039627
cat__nivel_educativo_Universitario    0.005724
cat__nivel_educativo_Técnico          0.005463
cat__nivel_educativo_Secundaria       0.005202
cat__nivel_educativo_Tecnólogo        0.004993
cat__ciudad_Ibagué                    0.003867
cat__ciudad_Cartagena                 0.003688
cat__ciudad_Villavicencio             0.003384
cat__nivel_educativo_Posgrado         0.003155
cat__ciudad_Cali                      0.002963
dtype: float64

In [37]:
def obtener_variable_original(nombre):
    if nombre.startswith("num__"):
        return nombre.replace("num__", "")

    if nombre.startswith("cat__nivel_educativo_"):
        return "nivel_educativo"

    if nombre.startswith("cat__ciudad_"):
        return "ciudad"

    return nombre


importancias_originales = (
    importancias_transformadas
    .groupby(
        [obtener_variable_original(x)
         for x in importancias_transformadas.index]
    )
    .sum()
    .sort_values(ascending=False)
)

importancias_originales = pd.DataFrame({
    "importancia": importancias_originales
})

display(importancias_originales)

,importancia
saldo_actual,0.355649
cuotas_pagadas,0.239167
score_externo,0.161697
cuotas_totales,0.078277
ciudad,0.053890
edad,0.044727
ingreso_mensual,0.039627
nivel_educativo,0.026967


In [38]:
top_10_variables = (
    importancias_originales
    .head(10)
    .index
    .tolist()
)

print("Variables seleccionadas para monitoreo:")
for i, variable in enumerate(top_10_variables, start=1):
    print(f"{i}. {variable}")

print(
    "\nNúmero de variables disponibles:",
    len(top_10_variables)
)

Variables seleccionadas para monitoreo:
1. saldo_actual
2. cuotas_pagadas
3. score_externo
4. cuotas_totales
5. ciudad
6. edad
7. ingreso_mensual
8. nivel_educativo

Número de variables disponibles: 8


## Variables prioritarias para monitoreo

Para identificar las variables que serán evaluadas en las pruebas de cambio de distribución se utilizó la importancia interna del `RandomForestClassifier`.

Debido a que el pipeline aplica One-Hot Encoding a `nivel_educativo` y `ciudad`, las importancias de sus categorías fueron agregadas para reconstruir la importancia de cada variable original.

Aunque la instrucción propone seleccionar las 10 variables más importantes, el modelo SOLARIS utiliza únicamente **8 variables predictoras originales**. Por tanto, se conservaron las 8 disponibles, ordenadas de mayor a menor importancia, sin incorporar variables adicionales que no formen parte del modelo.

Estas variables serán utilizadas como base para las pruebas de estabilidad y cambio de distribución de los pasos siguientes.

In [39]:
from scipy import stats

variables_ks = [
    var for var in top_10_variables
    if var in df_referencia.select_dtypes(include=np.number).columns
]

variables_no_ks = [
    var for var in top_10_variables
    if var not in variables_ks
]

print("Variables numéricas para KS:")
print(variables_ks)

print("\nVariables categóricas no evaluadas con KS:")
print(variables_no_ks)

Variables numéricas para KS:
['saldo_actual', 'cuotas_pagadas', 'score_externo', 'cuotas_totales', 'edad', 'ingreso_mensual']

Variables categóricas no evaluadas con KS:
['ciudad', 'nivel_educativo']


In [40]:
resultados_ks = []

for var in variables_ks:
    
    referencia = df_referencia[var].dropna()
    reciente = df_reciente[var].dropna()
    
    estadistico, p_valor = stats.ks_2samp(
        referencia,
        reciente
    )
    
    if p_valor < 0.01:
        alerta = "CRÍTICO"
    elif p_valor < 0.05:
        alerta = "REVISAR"
    else:
        alerta = "OK"
    
    resultados_ks.append({
        "variable": var,
        "n_referencia": len(referencia),
        "n_reciente": len(reciente),
        "estadistico_KS": round(estadistico, 4),
        "p_valor": round(p_valor, 6),
        "alerta": alerta
    })

df_ks = pd.DataFrame(resultados_ks)

df_ks

,variable,n_referencia,n_reciente,estadistico_KS,p_valor,alerta
0,saldo_actual,12164,136,0.0937,0.178593,OK
1,cuotas_pagadas,12164,136,0.0556,0.781354,OK
2,score_externo,11111,121,0.0551,0.839061,OK
3,cuotas_totales,12164,136,0.0353,0.994281,OK
4,edad,11327,126,0.0502,0.894598,OK
5,ingreso_mensual,12164,136,0.1101,0.071764,OK


In [41]:
for var in variables_no_ks:
    df_ks.loc[len(df_ks)] = {
        "variable": var,
        "n_referencia": df_referencia[var].notna().sum(),
        "n_reciente": df_reciente[var].notna().sum(),
        "estadistico_KS": np.nan,
        "p_valor": np.nan,
        "alerta": "NO APLICA - CATEGÓRICA"
    }

df_ks

,variable,n_referencia,n_reciente,estadistico_KS,p_valor,alerta
0,saldo_actual,12164,136,0.0937,0.178593,OK
1,cuotas_pagadas,12164,136,0.0556,0.781354,OK
2,score_externo,11111,121,0.0551,0.839061,OK
3,cuotas_totales,12164,136,0.0353,0.994281,OK
4,edad,11327,126,0.0502,0.894598,OK
5,ingreso_mensual,12164,136,0.1101,0.071764,OK
6,ciudad,12164,136,NaN,NaN,NO APLICA - CATEGÓRICA
7,nivel_educativo,12164,136,NaN,NaN,NO APLICA - CATEGÓRICA


In [42]:
REPORTS_PATH = PROJECT_ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

KS_PATH = REPORTS_PATH / "reporte_ks_test.csv"

df_ks.to_csv(
    KS_PATH,
    index=False
)

print("Reporte KS guardado en:", KS_PATH)

Reporte KS guardado en: c:\Users\anaso\Downloads\proyecto-solaris\reports\reporte_ks_test.csv


### Interpretación del KS Test

El Test de Kolmogórov-Smirnov no detectó cambios estadísticamente significativos en la distribución de las variables numéricas prioritarias del modelo entre los datos históricos de referencia y los distribuidores vinculados durante las últimas seis semanas.

Todas las variables evaluadas presentaron un p-valor superior a 0.05, por lo que fueron clasificadas como **OK** según los umbrales definidos para el análisis.

La variable con mayor estadístico KS fue `ingreso_mensual` (`KS = 0.1101`), con un p-valor de `0.0718`. Aunque es la variable que presenta la mayor diferencia entre ambas ventanas, el resultado no alcanza el umbral de significancia establecido y, por tanto, no genera una alerta en esta evaluación.

`saldo_actual`, una de las variables más importantes del modelo, también se mantuvo estable (`KS = 0.0937`, `p = 0.1786`). Las demás variables mostraron diferencias todavía menores y valores de p ampliamente superiores a 0.05.

Por tanto, con la evidencia disponible, **no se observa covariate shift significativo en las variables numéricas analizadas**. Sin embargo, esta conclusión debe interpretarse con cautela debido al tamaño reducido de la ventana reciente (136 observaciones).

Las variables categóricas `ciudad` y `nivel_educativo` no fueron evaluadas mediante KS, por lo que estos resultados no permiten descartar cambios de distribución en dichas variables.

Paso 4: Calcula el PSI para las variables con alerta

In [43]:
def calcular_psi(referencia, actual, n_bins=10):
    
    referencia = referencia.dropna()
    actual = actual.dropna()
    
    min_val = min(referencia.min(), actual.min())
    max_val = max(referencia.max(), actual.max())

    bins = np.linspace(
        min_val,
        max_val,
        n_bins + 1
    )

    ref_prop, _ = np.histogram(
        referencia,
        bins=bins
    )

    act_prop, _ = np.histogram(
        actual,
        bins=bins
    )

    ref_prop = np.where(
        ref_prop == 0,
        0.0001,
        ref_prop
    ) / len(referencia)

    act_prop = np.where(
        act_prop == 0,
        0.0001,
        act_prop
    ) / len(actual)

    psi = np.sum(
        (act_prop - ref_prop)
        * np.log(act_prop / ref_prop)
    )

    return round(psi, 4)

In [44]:
variables_alerta = df_ks.loc[
    df_ks["alerta"].isin(["REVISAR", "CRÍTICO"]),
    "variable"
].tolist()

print("Variables que requieren PSI:", variables_alerta)

Variables que requieren PSI: []


In [45]:
resultados_psi = []

for var in variables_alerta:
    
    psi = calcular_psi(
        df_referencia[var],
        df_reciente[var]
    )

    if psi < 0.1:
        clasificacion = "ESTABLE"
    elif psi < 0.2:
        clasificacion = "MODERADO"
    else:
        clasificacion = "CRÍTICO"

    resultados_psi.append({
        "variable": var,
        "psi": psi,
        "clasificacion": clasificacion
    })


df_psi = pd.DataFrame(
    resultados_psi,
    columns=[
        "variable",
        "psi",
        "clasificacion"
    ]
)

display(df_psi)

,variable,psi,clasificacion


In [46]:
PSI_PATH = REPORTS_PATH / "reporte_psi.csv"

df_psi.to_csv(
    PSI_PATH,
    index=False
)

print("Reporte PSI guardado en:", PSI_PATH)

Reporte PSI guardado en: c:\Users\anaso\Downloads\proyecto-solaris\reports\reporte_psi.csv


## Análisis PSI

El Population Stability Index (PSI) se definió como análisis complementario para aquellas variables que presentaran una alerta `REVISAR` o `CRÍTICO` en el Test de Kolmogórov-Smirnov.

En la evaluación realizada, ninguna de las variables numéricas prioritarias presentó un p-valor inferior a 0.05. Por tanto, **no se identificaron variables que requirieran el cálculo de PSI en esta ventana temporal**.

En consecuencia, el reporte `reporte_psi.csv` se genera sin registros. Esto no representa un error en el procedimiento, sino que refleja que el filtro previo basado en KS no detectó señales de cambio que requirieran una segunda evaluación mediante PSI.

Para futuras ventanas de monitoreo, si alguna variable presenta una alerta en KS, se aplicará PSI y se utilizarán los siguientes criterios:

- **PSI < 0.1:** ESTABLE
- **0.1 ≤ PSI < 0.2:** MODERADO
- **PSI ≥ 0.2:** CRÍTICO

Con la evidencia disponible en esta ventana, no se identifican señales de cambio de distribución que requieran escalamiento mediante PSI.

Paso 5: Valida el modelo en subgrupos demográficos

In [47]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

features = [
    "edad",
    "ingreso_mensual",
    "saldo_actual",
    "cuotas_pagadas",
    "cuotas_totales",
    "score_externo",
    "nivel_educativo",
    "ciudad"
]

df_modelo = df[
    df["estado_cuenta"].isin(["Al día", "Moroso"])
].copy()

df_modelo["mora"] = df_modelo["estado_cuenta"].map({
    "Al día": 0,
    "Moroso": 1
})

X = df_modelo[features].copy()
y = df_modelo["mora"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
    shuffle=True
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

df_eval = X_test.copy()

df_eval["y_real"] = y_test.values
df_eval["y_pred"] = y_pred
df_eval["y_prob"] = y_prob

X_test: (2460, 8)
y_test: (2460,)


In [48]:
df_eval["rango_edad"] = pd.cut(
    df_eval["edad"],
    bins=[0, 29, 39, 49, 59, 69, 120],
    labels=[
        "≤29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ]
)

In [49]:
mapa_ciudades = {
    "Bog": "Bogotá",
    "Bta": "Bogotá",

    "Mdl": "Medellín",
    "Med.": "Medellín",

    "Baq": "Barranquilla",
    "B/Quilla": "Barranquilla",
    "Barranqilla": "Barranquilla",

    "Bga": "Bucaramanga",
    "B/Manga": "Bucaramanga",

    "Cal": "Cali",
    "Santiago De Cali": "Cali",

    "San José De Cúcuta": "Cúcuta",

    "Mzl": "Manizales",

    "Pei": "Pereira",

    "Smr": "Santa Marta",
    "Sta Marta": "Santa Marta",

    "Villavo": "Villavicencio"
}

df_eval["ciudad_auditoria"] = (
    df_eval["ciudad"]
    .replace(mapa_ciudades)
)

Función para calcular métricas por grupo

In [50]:
def evaluar_subgrupos(df, variable_grupo):
    
    resultados = []

    for grupo in df[variable_grupo].dropna().unique():
        
        mascara = df[variable_grupo] == grupo
        
        if mascara.sum() > 30:
            
            y_real_grupo = df.loc[mascara, "y_real"]
            y_pred_grupo = df.loc[mascara, "y_pred"]
            y_prob_grupo = df.loc[mascara, "y_prob"]

            accuracy = accuracy_score(
                y_real_grupo,
                y_pred_grupo
            )

            f1 = f1_score(
                y_real_grupo,
                y_pred_grupo,
                zero_division=0
            )

            # AUC solamente puede calcularse si hay ambas clases
            if y_real_grupo.nunique() == 2:
                auc = roc_auc_score(
                    y_real_grupo,
                    y_prob_grupo
                )
            else:
                auc = np.nan

            resultados.append({
                "variable_sensible": variable_grupo,
                "subgrupo": grupo,
                "n": mascara.sum(),
                "accuracy": round(accuracy, 4),
                "auc_roc": round(auc, 4) if not np.isnan(auc) else np.nan,
                "f1": round(f1, 4)
            })

    return resultados



variables_subgrupo = [
    "rango_edad",
    "nivel_educativo",
    "ciudad_auditoria"
]

resultados_subgrupos = []

for variable in variables_subgrupo:
    resultados_subgrupos.extend(
        evaluar_subgrupos(
            df_eval,
            variable
        )
    )

df_subgrupos = pd.DataFrame(resultados_subgrupos)

display(df_subgrupos)

,variable_sensible,subgrupo,n,accuracy,auc_roc,f1
0,rango_edad,50-59,384,0.9271,0.9629,0.7500
1,rango_edad,≤29,454,0.9229,0.9569,0.6903
2,rango_edad,30-39,363,0.9532,0.9750,0.8317
3,rango_edad,40-49,334,0.9431,0.9742,0.7912
4,rango_edad,70+,370,0.9486,0.9501,0.7324
5,rango_edad,60-69,377,0.9337,0.9311,0.6914
6,nivel_educativo,Tecnólogo,429,0.9441,0.9650,0.7692
7,nivel_educativo,Secundaria,607,0.9292,0.9529,0.7034
8,nivel_educativo,Técnico,499,0.9439,0.9779,0.7667
9,nivel_educativo,Universitario,592,0.9341,0.9469,0.7547


In [51]:
resumen_brechas = []

for variable in variables_subgrupo:
    
    temp = df_subgrupos[
        df_subgrupos["variable_sensible"] == variable
    ].dropna(subset=["auc_roc"])

    if len(temp) >= 2:
        
        auc_max = temp["auc_roc"].max()
        auc_min = temp["auc_roc"].min()
        brecha_auc = auc_max - auc_min

        grupo_mejor = temp.loc[
            temp["auc_roc"].idxmax(),
            "subgrupo"
        ]

        grupo_peor = temp.loc[
            temp["auc_roc"].idxmin(),
            "subgrupo"
        ]

        riesgo = (
            "RIESGO"
            if brecha_auc > 0.05
            else "OK"
        )

        resumen_brechas.append({
            "variable": variable,
            "auc_max": round(auc_max, 4),
            "grupo_mejor_auc": grupo_mejor,
            "auc_min": round(auc_min, 4),
            "grupo_peor_auc": grupo_peor,
            "brecha_auc": round(brecha_auc, 4),
            "evaluacion": riesgo
        })

df_brechas_auc = pd.DataFrame(
    resumen_brechas
)

display(df_brechas_auc)

,variable,auc_max,grupo_mejor_auc,auc_min,grupo_peor_auc,brecha_auc,evaluacion
0,rango_edad,0.9750,30-39,0.9311,60-69,0.0439,OK
1,nivel_educativo,0.9779,Técnico,0.9384,Posgrado,0.0395,OK
2,ciudad_auditoria,0.9800,Medellín,0.9410,Cali,0.0390,OK


In [52]:
SUBGRUPOS_PATH = (
    REPORTS_PATH
    / "reporte_subgrupos.csv"
)

df_subgrupos.to_csv(
    SUBGRUPOS_PATH,
    index=False
)

print(
    "Reporte de subgrupos guardado en:",
    SUBGRUPOS_PATH
)


df_brechas_auc.to_csv(
    REPORTS_PATH / "reporte_brechas_auc.csv",
    index=False
)

Reporte de subgrupos guardado en: c:\Users\anaso\Downloads\proyecto-solaris\reports\reporte_subgrupos.csv


### Interpretación de la validación por subgrupos

La evaluación del modelo SOLARIS por subgrupos demográficos y potencialmente sensibles no identificó diferencias superiores a **0.05 en AUC-ROC**, que corresponde al umbral definido para reportar riesgo en esta actividad.

En los grupos de edad, el AUC-ROC varió entre **0.9311** para el grupo de 60-69 años y **0.9750** para el grupo de 30-39 años, con una brecha máxima de **0.0439**. Por tanto, no se identifica riesgo según el criterio establecido.

Para nivel educativo, el mejor desempeño se observó en el grupo `Técnico` con un AUC-ROC de **0.9779**, mientras que el menor se presentó en `Posgrado` con **0.9384**. La diferencia fue de **0.0395**, también inferior al umbral de riesgo.

En la evaluación por ciudad, el AUC-ROC más alto se obtuvo en Medellín (**0.9800**) y el menor en Cali (**0.9410**), con una brecha de **0.0390**. Este resultado tampoco supera el umbral establecido.

En consecuencia, **no se observa evidencia de diferencias relevantes en la capacidad de discriminación del modelo entre los subgrupos evaluados según el criterio de AUC-ROC definido en la actividad**.

Sin embargo, se observan diferencias mayores en la métrica F1 entre algunos subgrupos. Por ejemplo, por ciudad el F1 varía desde **0.6000 en Cartagena** hasta **0.8254 en Bogotá**. Aunque este comportamiento no activa la alerta definida por AUC-ROC, se recomienda mantenerlo bajo monitoreo porque indica que la calidad de la clasificación positiva puede variar entre grupos.

Paso 6: Mide demographic parity

## Demographic Parity

Se evaluó la paridad demográfica del modelo SOLARIS comparando la proporción de predicciones positivas (`Moroso = 1`) entre los diferentes subgrupos de edad, nivel educativo y ciudad.

Se mantuvieron únicamente grupos con más de 30 observaciones. Para cada variable sensible se calculó la diferencia entre la mayor y la menor tasa de predicción positiva.

Se estableció como criterio de alerta una diferencia superior a **0.10 (10 puntos porcentuales)** entre subgrupos.

In [53]:
variables_sensibles = [
    "rango_edad",
    "nivel_educativo",
    "ciudad_auditoria"
]
resultados_dp = []

for variable in variables_sensibles:

    for grupo in df_eval[variable].dropna().unique():

        mascara = df_eval[variable] == grupo

        # Mantenemos el criterio de mínimo 30 observaciones
        if mascara.sum() > 30:

            tasa_positiva = df_eval.loc[
                mascara,
                "y_pred"
            ].mean()

            resultados_dp.append({
                "variable_sensible": variable,
                "subgrupo": grupo,
                "n": mascara.sum(),
                "tasa_prediccion_positiva": round(
                    tasa_positiva,
                    4
                )
            })

df_demographic_parity = pd.DataFrame(
    resultados_dp
)

display(df_demographic_parity)

,variable_sensible,subgrupo,n,tasa_prediccion_positiva
0,rango_edad,50-59,384,0.1224
1,rango_edad,≤29,454,0.0991
2,rango_edad,30-39,363,0.1212
3,rango_edad,40-49,334,0.1138
4,rango_edad,70+,370,0.0757
5,rango_edad,60-69,377,0.0796
6,nivel_educativo,Tecnólogo,429,0.1026
7,nivel_educativo,Secundaria,607,0.0939
8,nivel_educativo,Técnico,499,0.1042
9,nivel_educativo,Universitario,592,0.1098


In [54]:
resultados_brecha_dp = []

for variable in variables_sensibles:

    temp = df_demographic_parity[
        df_demographic_parity["variable_sensible"] == variable
    ]

    tasa_max = temp[
        "tasa_prediccion_positiva"
    ].max()

    tasa_min = temp[
        "tasa_prediccion_positiva"
    ].min()

    brecha = tasa_max - tasa_min

    grupo_max = temp.loc[
        temp["tasa_prediccion_positiva"].idxmax(),
        "subgrupo"
    ]

    grupo_min = temp.loc[
        temp["tasa_prediccion_positiva"].idxmin(),
        "subgrupo"
    ]

    evaluacion = (
        "ALERTA"
        if brecha > 0.10
        else "OK"
    )

    resultados_brecha_dp.append({
        "variable_sensible": variable,
        "grupo_mayor_tasa": grupo_max,
        "tasa_max": round(tasa_max, 4),
        "grupo_menor_tasa": grupo_min,
        "tasa_min": round(tasa_min, 4),
        "brecha_dp": round(brecha, 4),
        "evaluacion": evaluacion
    })

df_brechas_dp = pd.DataFrame(
    resultados_brecha_dp
)

display(df_brechas_dp)

,variable_sensible,grupo_mayor_tasa,tasa_max,grupo_menor_tasa,tasa_min,brecha_dp,evaluacion
0,rango_edad,50-59,0.1224,70+,0.0757,0.0467,OK
1,nivel_educativo,Universitario,0.1098,Posgrado,0.0904,0.0194,OK
2,ciudad_auditoria,Manizales,0.1284,Santa Marta,0.0657,0.0627,OK


In [55]:
df_demographic_parity.to_csv(
    REPORTS_PATH / "reporte_demographic_parity.csv",
    index=False
)

df_brechas_dp.to_csv(
    REPORTS_PATH / "reporte_brechas_demographic_parity.csv",
    index=False
)

### Interpretación de Demographic Parity

La evaluación de Demographic Parity no identificó brechas superiores al umbral de **0.10 (10 puntos porcentuales)** en ninguna de las variables sensibles analizadas.

Para los rangos de edad, la mayor tasa de predicción de mora se observó en el grupo de **50-59 años** con **0.1224**, mientras que la menor correspondió al grupo de **70+** con **0.0757**. La diferencia máxima fue de **0.0467**, por lo que la variable se clasifica como **OK**.

En nivel educativo, la mayor tasa se presentó en el grupo **Universitario** con **0.1098**, y la menor en **Posgrado** con **0.0904**. La brecha fue de **0.0194**, también dentro del rango aceptable.

Por ciudad, la mayor tasa de predicción positiva correspondió a **Manizales** con **0.1284**, mientras que la menor se observó en **Santa Marta** con **0.0657**. La diferencia fue de **0.0627**, siendo la brecha más alta entre las variables evaluadas, aunque permanece por debajo del umbral de alerta.

En consecuencia, **no se detectan alertas de Demographic Parity según el criterio definido para esta actividad**. Sin embargo, las diferencias observadas, especialmente entre ciudades, deben continuar monitoreándose en futuras ventanas de evaluación.

Paso 7: Mide equalized odds


## Equalized Odds

Se evaluó Equalized Odds comparando las tasas de verdaderos positivos (TPR) y falsos positivos (FPR) entre los subgrupos de edad, nivel educativo y ciudad.

Solo se analizaron grupos con más de 30 observaciones.

- **TPR (True Positive Rate):** proporción de clientes realmente morosos que fueron correctamente identificados como morosos por el modelo.
- **FPR (False Positive Rate):** proporción de clientes realmente al día que fueron clasificados incorrectamente como morosos.

Posteriormente, se calcularon las diferencias máximas de TPR y FPR entre los subgrupos de cada variable sensible para identificar posibles diferencias en la distribución de los errores del modelo.

In [56]:
from sklearn.metrics import confusion_matrix


variables_sensibles = [
    "rango_edad",
    "nivel_educativo",
    "ciudad_auditoria"
]

print("Variables sensibles evaluadas:")
print(variables_sensibles)


resultados_eo = []

for variable in variables_sensibles:

    for grupo in df_eval[variable].dropna().unique():

        mascara = df_eval[variable] == grupo

        # Solo evaluar grupos con más de 30 observaciones
        if mascara.sum() > 30:

            y_real_grupo = df_eval.loc[
                mascara,
                "y_real"
            ]

            y_pred_grupo = df_eval.loc[
                mascara,
                "y_pred"
            ]

            tn, fp, fn, tp = confusion_matrix(
                y_real_grupo,
                y_pred_grupo,
                labels=[0, 1]
            ).ravel()

            # True Positive Rate
            tpr = (
                tp / (tp + fn)
                if (tp + fn) > 0
                else np.nan
            )

            # False Positive Rate
            fpr = (
                fp / (fp + tn)
                if (fp + tn) > 0
                else np.nan
            )

            resultados_eo.append({
                "variable_sensible": variable,
                "subgrupo": grupo,
                "n": mascara.sum(),
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
                "TPR": round(tpr, 4),
                "FPR": round(fpr, 4)
            })


df_equalized_odds = pd.DataFrame(
    resultados_eo
)

display(df_equalized_odds)

Variables sensibles evaluadas:
['rango_edad', 'nivel_educativo', 'ciudad_auditoria']


,variable_sensible,subgrupo,n,tn,fp,fn,tp,TPR,FPR
0,rango_edad,50-59,384,314,5,23,42,0.6462,0.0157
1,rango_edad,≤29,454,380,6,29,39,0.5735,0.0155
2,rango_edad,30-39,363,304,2,15,42,0.7368,0.0065
3,rango_edad,40-49,334,279,2,17,36,0.6792,0.0071
4,rango_edad,70+,370,325,2,17,26,0.6047,0.0061
5,rango_edad,60-69,377,324,2,23,28,0.5490,0.0061
6,nivel_educativo,Tecnólogo,429,365,4,20,40,0.6667,0.0108
7,nivel_educativo,Secundaria,607,513,6,37,51,0.5795,0.0116
8,nivel_educativo,Técnico,499,425,6,22,46,0.6765,0.0139
9,nivel_educativo,Universitario,592,493,5,34,60,0.6383,0.0100


In [57]:
resumen_equalized_odds = []

for variable in variables_sensibles:

    temp = df_equalized_odds[
        df_equalized_odds["variable_sensible"] == variable
    ].copy()

    temp_tpr = temp.dropna(
        subset=["TPR"]
    )

    temp_fpr = temp.dropna(
        subset=["FPR"]
    )

    # Valores máximos y mínimos de TPR
    tpr_max = temp_tpr["TPR"].max()
    tpr_min = temp_tpr["TPR"].min()

    brecha_tpr = (
        tpr_max - tpr_min
    )

    # Grupos con mayor y menor TPR
    grupo_tpr_max = temp_tpr.loc[
        temp_tpr["TPR"].idxmax(),
        "subgrupo"
    ]

    grupo_tpr_min = temp_tpr.loc[
        temp_tpr["TPR"].idxmin(),
        "subgrupo"
    ]

    # Valores máximos y mínimos de FPR
    fpr_max = temp_fpr["FPR"].max()
    fpr_min = temp_fpr["FPR"].min()

    brecha_fpr = (
        fpr_max - fpr_min
    )

    # Grupos con mayor y menor FPR
    grupo_fpr_max = temp_fpr.loc[
        temp_fpr["FPR"].idxmax(),
        "subgrupo"
    ]

    grupo_fpr_min = temp_fpr.loc[
        temp_fpr["FPR"].idxmin(),
        "subgrupo"
    ]

    resumen_equalized_odds.append({
        "variable_sensible": variable,

        "TPR_max": round(tpr_max, 4),
        "grupo_TPR_max": grupo_tpr_max,

        "TPR_min": round(tpr_min, 4),
        "grupo_TPR_min": grupo_tpr_min,

        "brecha_TPR": round(
            brecha_tpr,
            4
        ),

        "FPR_max": round(fpr_max, 4),
        "grupo_FPR_max": grupo_fpr_max,

        "FPR_min": round(fpr_min, 4),
        "grupo_FPR_min": grupo_fpr_min,

        "brecha_FPR": round(
            brecha_fpr,
            4
        )
    })


df_brechas_eo = pd.DataFrame(
    resumen_equalized_odds
)

display(df_brechas_eo)

,variable_sensible,TPR_max,grupo_TPR_max,TPR_min,grupo_TPR_min,brecha_TPR,FPR_max,grupo_FPR_max,FPR_min,grupo_FPR_min,brecha_FPR
0,rango_edad,0.7368,30-39,0.5490,60-69,0.1878,0.0157,50-59,0.0061,70+,0.0096
1,nivel_educativo,0.6765,Técnico,0.5517,Posgrado,0.1248,0.0139,Técnico,0.0063,Posgrado,0.0076
2,ciudad_auditoria,0.7333,Medellín,0.4800,Cartagena,0.2533,0.0176,Villavicencio,0.0000,Ibagué,0.0176


In [58]:
df_equalized_odds.to_csv(
    REPORTS_PATH / "reporte_equalized_odds.csv",
    index=False
)

df_brechas_eo.to_csv(
    REPORTS_PATH / "reporte_brechas_equalized_odds.csv",
    index=False
)

print(
    "Reportes de Equalized Odds guardados correctamente."
)

Reportes de Equalized Odds guardados correctamente.


### Interpretación de Equalized Odds

La evaluación de Equalized Odds evidencia diferencias relevantes principalmente en la tasa de verdaderos positivos (TPR), mientras que las diferencias en la tasa de falsos positivos (FPR) son reducidas.

Para `rango_edad`, la TPR más alta se observa en el grupo de **30-39 años** con **0.7368**, mientras que la más baja corresponde al grupo de **60-69 años** con **0.5490**. Esto representa una brecha de **0.1878 (18.78 puntos porcentuales)**. En contraste, la brecha máxima de FPR es de solo **0.0096**.

En `nivel_educativo`, el grupo `Técnico` presenta la TPR más alta (**0.6765**) y `Posgrado` la más baja (**0.5517**), con una diferencia de **0.1248 (12.48 puntos porcentuales)**. La diferencia máxima en FPR es considerablemente menor (**0.0076**).

La mayor disparidad se observa por ciudad. Medellín alcanza una TPR de **0.7333**, mientras que Cartagena presenta una TPR de **0.4800**, generando una brecha de **0.2533 (25.33 puntos porcentuales)**. Esto significa que el modelo identifica correctamente a una proporción considerablemente mayor de clientes morosos en Medellín que en Cartagena. La brecha de FPR por ciudad es de **0.0176**, por lo que la principal diferencia se concentra en la capacidad del modelo para detectar verdaderos morosos y no en la generación de falsos positivos.

Por tanto, aunque las tasas de falsos positivos son similares entre los grupos, **sí existe una señal de trato desigual en términos de TPR**, especialmente por ciudad, seguida por rango de edad y nivel educativo. Esta diferencia debe reportarse como un riesgo de fairness y mantenerse bajo monitoreo.

Paso 8: Documenta cada riesgo identificado

In [59]:
riesgos = [
    {
        "Variable_Dimension": "ciudad",
        "Tipo_Riesgo": "Equalized Odds - brecha TPR",
        "Nivel_Severidad": "ALTO",
        "Evidencia_Cuantitativa": (
            "TPR Medellín=0.7333; "
            "TPR Cartagena=0.4800; "
            "brecha TPR=0.2533; "
            "brecha FPR=0.0176"
        ),
        "Impacto_Potencial": (
            "El modelo detecta una proporción considerablemente menor "
            "de clientes realmente morosos en algunas ciudades, "
            "especialmente Cartagena."
        )
    },
    {
        "Variable_Dimension": "rango_edad",
        "Tipo_Riesgo": "Equalized Odds - brecha TPR",
        "Nivel_Severidad": "MEDIO",
        "Evidencia_Cuantitativa": (
            "TPR 30-39=0.7368; "
            "TPR 60-69=0.5490; "
            "brecha TPR=0.1878; "
            "brecha FPR=0.0096"
        ),
        "Impacto_Potencial": (
            "La capacidad para detectar correctamente la mora "
            "varía entre grupos de edad."
        )
    },
    {
        "Variable_Dimension": "nivel_educativo",
        "Tipo_Riesgo": "Equalized Odds - brecha TPR",
        "Nivel_Severidad": "MEDIO",
        "Evidencia_Cuantitativa": (
            "TPR Técnico=0.6765; "
            "TPR Posgrado=0.5517; "
            "brecha TPR=0.1248; "
            "brecha FPR=0.0076"
        ),
        "Impacto_Potencial": (
            "La sensibilidad del modelo para detectar mora "
            "es diferente entre niveles educativos."
        )
    }
]

df_riesgos = pd.DataFrame(riesgos)

display(df_riesgos)

,Variable_Dimension,Tipo_Riesgo,Nivel_Severidad,Evidencia_Cuantitativa,Impacto_Potencial
0,ciudad,Equalized Odds - brecha TPR,ALTO,TPR Medellín=0.7333; TPR Cartagena=0.4800; bre...,El modelo detecta una proporción considerablem...
1,rango_edad,Equalized Odds - brecha TPR,MEDIO,TPR 30-39=0.7368; TPR 60-69=0.5490; brecha TPR...,La capacidad para detectar correctamente la mo...
2,nivel_educativo,Equalized Odds - brecha TPR,MEDIO,TPR Técnico=0.6765; TPR Posgrado=0.5517; brech...,La sensibilidad del modelo para detectar mora ...


In [60]:
RIESGOS_PATH = REPORTS_PATH / "reporte_riesgos.csv"

df_riesgos.to_csv(
    RIESGOS_PATH,
    index=False
)

print("Reporte de riesgos guardado en:", RIESGOS_PATH)

Reporte de riesgos guardado en: c:\Users\anaso\Downloads\proyecto-solaris\reports\reporte_riesgos.csv


### Conclusión del registro de riesgos

Las pruebas de covariate shift no generaron alertas y tampoco se identificaron brechas superiores al umbral establecido en AUC-ROC o Demographic Parity. Sin embargo, el análisis de Equalized Odds sí mostró diferencias relevantes en la tasa de verdaderos positivos (TPR).

La principal prioridad corresponde a la dimensión geográfica, donde la brecha de TPR alcanza **25.33 puntos porcentuales** entre Medellín y Cartagena. También se identificaron diferencias de **18.78 puntos porcentuales** por rango de edad y **12.48 puntos porcentuales** por nivel educativo.

Estas diferencias no demuestran por sí solas discriminación, pero constituyen señales de riesgo que deben ser comunicadas, monitoreadas y analizadas antes del despliegue del modelo.

Paso 9: Propón mitigaciones específicas


## Mitigaciones propuestas

| Riesgo identificado | Mitigación concreta | Responsable | Plazo | Criterio de seguimiento |
|---|---|---|---|---|
| **Ciudad — brecha TPR = 0.2533** | Revisar la representación de las ciudades en entrenamiento, normalizar definitivamente las categorías duplicadas de `ciudad` y evaluar reweighting o resampling para los grupos con menor sensibilidad. Reentrenar una versión candidata y volver a calcular Equalized Odds antes de desplegar. | Científico de Datos del proyecto, con revisión de la Líder Técnica | **2 semanas antes del despliegue** | Reducir la brecha TPR entre ciudades respecto al 0.2533 actual. Mantener monitoreo mensual y generar alerta si la brecha supera **0.10**. |
| **Rango de edad — brecha TPR = 0.1878** | Analizar si los grupos de edad con menor TPR están subrepresentados o presentan patrones distintos. Probar reponderación de muestras durante el entrenamiento y comparar desempeño global y fairness antes de aceptar el cambio. | Científico de Datos | **2 semanas** | Recalcular TPR/FPR por rango de edad. Revisar especialmente la diferencia entre 30-39 y 60-69 años y escalar si continúa por encima de **0.10**. |
| **Nivel educativo — brecha TPR = 0.1248** | Evaluar la distribución de observaciones por nivel educativo y probar reweighting/resampling para reducir la diferencia de sensibilidad. Discutir con el cliente si `nivel_educativo` debe mantenerse como predictor y cuál definición de fairness se priorizará para SOLARIS. | Científico de Datos + Líder Técnica + representante del cliente | **Antes de producción, máximo 2 semanas** | Volver a calcular Equalized Odds después de la mitigación y verificar si la brecha TPR disminuye respecto al **0.1248** actual. |

### Estrategia de mitigación

Las acciones propuestas priorizan la reducción de las diferencias observadas en la tasa de verdaderos positivos sin asumir que eliminar una variable sensible garantiza automáticamente un modelo justo.

En primer lugar, se propone revisar la representación de los grupos y probar técnicas de reweighting o resampling antes de modificar directamente la lógica de decisión del modelo. Cualquier nueva versión deberá compararse con el modelo actual tanto en desempeño predictivo como en métricas de fairness.

Si las brechas continúan después del reentrenamiento, podrá evaluarse una estrategia de postprocesamiento, como ajustes de umbral, únicamente después de acordar con el cliente qué definición de fairness se priorizará y documentar formalmente esa decisión.

Finalmente, Equalized Odds deberá incorporarse al monitoreo periódico del modelo. Se propone revisar mensualmente TPR y FPR por grupo y generar una alerta cuando alguna brecha supere 0.10.

Paso 10: Genera el reporte en formato PDF, HTML o Markdown

# Paso 10: Reporte final de Validación y Riesgos

Se consolidan los resultados obtenidos en las pruebas de estabilidad temporal, validación por subgrupos, fairness, registro de riesgos y plan de mitigaciones en un único reporte ejecutivo.

In [61]:
# ============================================================
# PASO 10 - REPORTE FINAL DE VALIDACIÓN Y RIESGOS SOLARIS
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# 1. TABLA DE MITIGACIONES
# ============================================================

mitigaciones = [
    {
        "Riesgo_identificado": "Ciudad - brecha TPR = 0.2533",

        "Mitigacion_concreta": (
            "Normalizar definitivamente las categorías duplicadas de ciudad, "
            "revisar la representación de cada ciudad en el conjunto de "
            "entrenamiento y probar técnicas de reweighting o resampling. "
            "Posteriormente, reentrenar una versión candidata del modelo y "
            "volver a calcular Equalized Odds antes del despliegue."
        ),

        "Responsable": (
            "Científico de Datos del proyecto + revisión de Líder Técnica"
        ),

        "Plazo": (
            "2 semanas antes del despliegue"
        ),

        "Criterio_seguimiento": (
            "Reducir la brecha respecto al 0.2533 actual. "
            "Realizar monitoreo mensual y generar una alerta si "
            "la brecha TPR supera 0.10."
        )
    },

    {
        "Riesgo_identificado": "Rango de edad - brecha TPR = 0.1878",

        "Mitigacion_concreta": (
            "Analizar la representación y los patrones del grupo de 60-69 años, "
            "probar reponderación de muestras durante el entrenamiento y comparar "
            "el desempeño global y las métricas de fairness con el modelo actual "
            "antes de aceptar una nueva versión."
        ),

        "Responsable": (
            "Científico de Datos"
        ),

        "Plazo": (
            "2 semanas"
        ),

        "Criterio_seguimiento": (
            "Recalcular TPR y FPR por rango de edad y escalar el riesgo "
            "si la brecha continúa por encima de 0.10."
        )
    },

    {
        "Riesgo_identificado": (
            "Nivel educativo - brecha TPR = 0.1248"
        ),

        "Mitigacion_concreta": (
            "Revisar la distribución de observaciones por nivel educativo, "
            "probar técnicas de reweighting o resampling y discutir con el "
            "cliente si la variable debe mantenerse como predictor y qué "
            "definición de fairness debe priorizarse para SOLARIS."
        ),

        "Responsable": (
            "Científico de Datos + Líder Técnica + representante del cliente"
        ),

        "Plazo": (
            "Antes de producción, máximo 2 semanas"
        ),

        "Criterio_seguimiento": (
            "Recalcular Equalized Odds después de la mitigación y verificar "
            "que la brecha TPR disminuya respecto al 0.1248 actual."
        )
    }
]


df_mitigaciones = pd.DataFrame(
    mitigaciones
)


# ============================================================
# 2. FUNCIÓN PARA CONVERTIR DATAFRAMES A HTML
# ============================================================

def tabla_html(
    df,
    decimales=4
):

    if df is None or df.empty:

        return """
        <p class="nota">
            No se generaron registros para esta sección.
        </p>
        """

    return df.to_html(
        index=False,
        border=0,
        classes="tabla",
        na_rep="—",
        float_format=lambda x: f"{x:.{decimales}f}"
    )


# ============================================================
# 3. SECCIÓN PSI
# ============================================================

if df_psi.empty:

    psi_html = """
    <p>
        Ninguna variable presentó una alerta
        <strong>REVISAR</strong> o <strong>CRÍTICO</strong>
        en el KS Test.
    </p>

    <p>
        Por esta razón, de acuerdo con el procedimiento definido
        para la actividad, no fue necesario calcular PSI para esta
        ventana temporal.
    </p>

    <p>
        Para futuras evaluaciones se utilizarán los siguientes criterios:
    </p>

    <ul>
        <li>PSI &lt; 0.1: ESTABLE</li>
        <li>0.1 ≤ PSI &lt; 0.2: MODERADO</li>
        <li>PSI ≥ 0.2: CRÍTICO</li>
    </ul>
    """

else:

    psi_html = tabla_html(
        df_psi
    )


# ============================================================
# 4. ESTILOS DEL REPORTE
# ============================================================

estilos = """
<style>

body {
    font-family: Arial, Helvetica, sans-serif;
    color: #1f2937;
    background-color: #f3f4f6;
    line-height: 1.55;
    margin: 0;
}

.contenedor {
    max-width: 1200px;
    margin: 30px auto;
    background: white;
    padding: 45px 55px;
    box-shadow: 0px 2px 14px rgba(0,0,0,0.08);
}

h1 {
    color: #17365d;
    margin-bottom: 5px;
}

.subtitulo {
    color: #6b7280;
    margin-bottom: 35px;
}

h2 {
    color: #17365d;
    margin-top: 38px;
    padding-bottom: 7px;
    border-bottom: 2px solid #dbe2ea;
}

h3 {
    color: #263f5f;
    margin-top: 25px;
}

.tabla {
    width: 100%;
    border-collapse: collapse;
    margin-top: 15px;
    margin-bottom: 25px;
    font-size: 13px;
}

.tabla th {
    background-color: #eaf0f6;
    border: 1px solid #d1d5db;
    padding: 8px;
    text-align: left;
}

.tabla td {
    border: 1px solid #d1d5db;
    padding: 8px;
    vertical-align: top;
}

.tabla tr:nth-child(even) {
    background-color: #fafafa;
}

.resumen {
    background-color: #f4f7fa;
    border-left: 5px solid #17365d;
    padding: 18px 22px;
    margin: 20px 0;
}

.alerta {
    background-color: #fff2f2;
    border-left: 5px solid #b91c1c;
    padding: 15px 20px;
    margin: 20px 0;
}

.ok {
    background-color: #f0f9f4;
    border-left: 5px solid #15803d;
    padding: 15px 20px;
    margin: 20px 0;
}

.nota {
    color: #6b7280;
    font-style: italic;
}

.limitacion {
    background-color: #fffbea;
    border-left: 5px solid #ca8a04;
    padding: 15px 20px;
    margin: 20px 0;
}

code {
    background-color: #edf0f3;
    padding: 2px 5px;
    border-radius: 3px;
}

@media print {

    body {
        background: white;
    }

    .contenedor {
        box-shadow: none;
        margin: 0;
        max-width: none;
    }
}

</style>
"""


# ============================================================
# 5. CONSTRUIR REPORTE HTML
# ============================================================

html_reporte = f"""
<!DOCTYPE html>

<html lang="es">

<head>

<meta charset="UTF-8">

<title>
Reporte de Validación y Riesgos - SOLARIS
</title>

{estilos}

</head>


<body>

<div class="contenedor">


<h1>
Reporte de Validación y Riesgos
</h1>

<p class="subtitulo">
Proyecto SOLARIS · QUANTUM ANALYTICS · Módulo 4
</p>



<!-- ====================================================== -->
<!-- 1. RESUMEN EJECUTIVO -->
<!-- ====================================================== -->

<h2>
1. Resumen ejecutivo
</h2>


<div class="resumen">

<p>
La validación temporal del modelo SOLARIS no encontró evidencia
estadísticamente significativa de <i>covariate shift</i> en las seis
variables numéricas prioritarias.

Todas presentaron valores p superiores a 0.05 en el KS Test,
por lo que ninguna activó el análisis mediante PSI.

La ventana reciente estuvo compuesta por
<strong>136 distribuidores</strong> vinculados entre el
20 de mayo y el 30 de junio de 2025, frente a
<strong>12.164 observaciones históricas</strong> de referencia.

Debido al tamaño reducido de la ventana reciente,
los resultados deben interpretarse como evidencia de monitoreo
y no como prueba definitiva de ausencia de degradación.
</p>


<p>
En la validación por subgrupos no se identificaron diferencias
de AUC-ROC superiores a 0.05 ni alertas de Demographic Parity
superiores a 0.10.

Sin embargo, Equalized Odds reveló diferencias importantes
en la capacidad del modelo para identificar correctamente
a los clientes realmente morosos.

La brecha de TPR alcanzó
<strong>0.2533 por ciudad</strong>,
<strong>0.1878 por rango de edad</strong> y
<strong>0.1248 por nivel educativo</strong>.

Estos resultados constituyen los principales riesgos identificados
y requieren acciones de mitigación y monitoreo antes del despliegue.
</p>

</div>



<!-- ====================================================== -->
<!-- 2. DATOS ANALIZADOS -->
<!-- ====================================================== -->

<h2>
2. Datos analizados
</h2>


<p>
El análisis consideró únicamente distribuidores con estado
<strong>Al día</strong> o <strong>Moroso</strong>,
manteniendo la misma población utilizada por el modelo y
excluyendo los registros con estado
<strong>Cancelado</strong>.
</p>


<ul>

<li>
<strong>Datos de referencia:</strong>
12.164 observaciones entre el
1 de enero de 2015 y el 19 de mayo de 2025.
</li>

<li>
<strong>Datos recientes:</strong>
136 observaciones entre el
20 de mayo y el 30 de junio de 2025.
</li>

<li>
<strong>Fecha de corte:</strong>
19 de mayo de 2025.
</li>

<li>
<strong>Ventana reciente:</strong>
6 semanas.
</li>

<li>
<strong>Total analizado:</strong>
12.300 observaciones.
</li>

</ul>


<p class="nota">

La variable temporal disponible es
<code>fecha_vinculacion</code>.

Por tanto, este análisis permite estudiar cambios en el perfil
de los distribuidores vinculados recientemente frente al histórico,
pero no demuestra cambios en el comportamiento posterior de
clientes ya existentes.

</p>



<!-- ====================================================== -->
<!-- 3. DISTRIBUTIONAL SHIFT -->
<!-- ====================================================== -->

<h2>
3. Resultados de Distributional Shift
</h2>


<h3>
KS Test
</h3>


<p>
El Test de Kolmogórov-Smirnov se aplicó a las variables numéricas
prioritarias del modelo.

Los criterios utilizados fueron:
</p>

<ul>

<li>
<strong>CRÍTICO:</strong>
p &lt; 0.01
</li>

<li>
<strong>REVISAR:</strong>
p &lt; 0.05
</li>

<li>
<strong>OK:</strong>
p ≥ 0.05
</li>

</ul>


{tabla_html(df_ks, 6)}


<div class="ok">

<strong>Resultado:</strong>

No se encontró evidencia estadísticamente significativa de
covariate shift en las variables numéricas evaluadas.

La variable más cercana al umbral fue
<code>ingreso_mensual</code>,
con:

<strong>KS = 0.1101</strong>

y

<strong>p = 0.071764</strong>.

</div>



<h3>
Population Stability Index (PSI)
</h3>


{psi_html}



<!-- ====================================================== -->
<!-- 4. VALIDACIÓN POR SUBGRUPOS -->
<!-- ====================================================== -->

<h2>
4. Validación por subgrupos
</h2>


<p>
El desempeño del modelo fue evaluado sobre el conjunto de prueba
independiente compuesto por
<strong>2.460 observaciones</strong>.

Se calcularon las métricas
Accuracy, AUC-ROC y F1
para los siguientes grupos:
</p>

<ul>

<li>Rango de edad</li>
<li>Nivel educativo</li>
<li>Ciudad</li>

</ul>


<p>
Solo se consideraron grupos con más de
<strong>30 observaciones</strong>.
</p>



<h3>
Brechas de AUC-ROC
</h3>


{tabla_html(df_brechas_auc)}


<div class="ok">

Ninguna dimensión presentó una diferencia de
AUC-ROC superior al umbral de
<strong>0.05</strong>.

La mayor brecha fue de
<strong>0.0439</strong>
por rango de edad.

</div>



<h3>
Resultados completos por subgrupo
</h3>


{tabla_html(df_subgrupos)}


<p>
Aunque las brechas de AUC-ROC permanecen dentro del criterio
establecido, se observaron diferencias mayores en F1.

Por ciudad, por ejemplo,
<strong>Cartagena presenta F1 = 0.6000</strong>,
mientras
<strong>Bogotá alcanza F1 = 0.8254</strong>.

Este comportamiento no activa la alerta definida por AUC-ROC,
pero indica que la calidad de la clasificación positiva puede
variar entre grupos.
</p>



<!-- ====================================================== -->
<!-- 5. FAIRNESS -->
<!-- ====================================================== -->

<h2>
5. Análisis de Fairness
</h2>



<!-- ---------------- DEMOGRAPHIC PARITY ---------------- -->

<h3>
Demographic Parity
</h3>


<p>
La clase positiva corresponde a:

<strong>Moroso = 1</strong>.

Por tanto, la tasa de predicción positiva representa la proporción
de distribuidores clasificados como morosos en cada grupo.
</p>


<p>
Se estableció como criterio de alerta una diferencia superior a:

<strong>0.10 (10 puntos porcentuales)</strong>.
</p>


{tabla_html(df_brechas_dp)}


<div class="ok">

<strong>Resultado:</strong>

No se identificaron alertas de Demographic Parity.

La mayor brecha fue por ciudad, con:

<strong>0.0627</strong>,

valor inferior al umbral de 0.10.

</div>



<h3>
Tasas completas de Demographic Parity
</h3>


{tabla_html(df_demographic_parity)}



<!-- ---------------- EQUALIZED ODDS ---------------- -->

<h3>
Equalized Odds
</h3>


<p>
Se compararon las tasas de verdaderos positivos
<strong>(TPR)</strong>
y falsos positivos
<strong>(FPR)</strong>
entre grupos.

Las diferencias de FPR son reducidas,
pero se detectaron brechas considerables en TPR.
</p>


<div class="limitacion">

<strong>Consideración sobre el tamaño de los grupos:</strong>

<p>
Aunque todos los subgrupos incluidos superan las
30 observaciones, el número de casos realmente morosos
dentro de algunos grupos es relativamente reducido.
</p>

<p>
Por ejemplo, Cartagena contiene
<strong>25 clientes realmente morosos</strong>:

12 verdaderos positivos + 13 falsos negativos.

Medellín contiene
<strong>30 clientes realmente morosos</strong>:

22 verdaderos positivos + 8 falsos negativos.
</p>

<p>
Por esta razón, las diferencias de TPR deben interpretarse como
<strong>señales de riesgo que requieren seguimiento</strong>
y no como evidencia definitiva de discriminación.

Cambios de pocos casos pueden producir variaciones relevantes
en la TPR cuando el número de positivos reales por subgrupo
es reducido.
</p>

</div>


{tabla_html(df_brechas_eo)}


<div class="alerta">

<strong>Principal riesgo identificado:</strong>

<p>
Por ciudad,
<strong>Medellín presenta TPR = 0.7333</strong>,
mientras
<strong>Cartagena presenta TPR = 0.4800</strong>.
</p>

<p>
La diferencia corresponde a una brecha de:
</p>

<strong>
0.2533
(25.33 puntos porcentuales)
</strong>.

</div>



<h3>
Resultados completos de Equalized Odds
</h3>


{tabla_html(df_equalized_odds)}



<!-- ====================================================== -->
<!-- 6. RIESGOS -->
<!-- ====================================================== -->

<h2>
6. Tabla de riesgos
</h2>


<p>
Para priorización interna de este reporte se utiliza el siguiente
criterio:
</p>


<ul>

<li>
<strong>ALTO:</strong>
brecha TPR superior a 0.20.
</li>

<li>
<strong>MEDIO:</strong>
brecha TPR entre 0.10 y 0.20.
</li>

</ul>


<p class="nota">

Este criterio se utiliza únicamente para priorizar acciones
dentro del reporte y no representa un estándar universal
de fairness.

</p>


{tabla_html(df_riesgos)}



<!-- ====================================================== -->
<!-- 7. MITIGACIONES -->
<!-- ====================================================== -->

<h2>
7. Plan de mitigaciones
</h2>


{tabla_html(df_mitigaciones)}



<h3>
Estrategia de seguimiento
</h3>


<p>
Las acciones de mitigación seguirán el ciclo:
</p>


<p>

<strong>
medir → intervenir → reentrenar → volver a medir → aprobar
</strong>

</p>


<p>
Equalized Odds deberá incorporarse al monitoreo periódico
del modelo.

Se propone revisar mensualmente TPR y FPR por grupo y
generar una alerta cuando alguna brecha supere
<strong>0.10</strong>.
</p>


<p>
Si las diferencias persisten después de aplicar técnicas de
reweighting o resampling, podrá evaluarse una estrategia de
postprocesamiento, como ajustes de umbral.

Cualquier modificación deberá ser analizada junto con el cliente
y documentada antes de implementar una nueva versión del modelo.
</p>



<!-- ====================================================== -->
<!-- CONCLUSIÓN -->
<!-- ====================================================== -->

<h2>
Conclusión general
</h2>


<p>
Los análisis realizados no identifican señales estadísticamente
significativas de cambio de distribución en las variables numéricas
evaluadas ni diferencias de capacidad discriminativa superiores
al umbral establecido de AUC-ROC entre los subgrupos analizados.
</p>


<p>
Demographic Parity tampoco genera alertas según el criterio
de 10 puntos porcentuales definido para la actividad.
</p>


<p>
El principal punto de atención corresponde a Equalized Odds,
donde se identifican diferencias en la sensibilidad del modelo
entre ciudades, grupos de edad y niveles educativos.
</p>


<p>
Estas diferencias
<strong>
no demuestran por sí solas discriminación
</strong>,
pero constituyen riesgos de fairness que deben ser
mitigados, documentados y monitoreados antes de la puesta
en producción.
</p>



<!-- ====================================================== -->
<!-- LIMITACIONES -->
<!-- ====================================================== -->

<h3>
Limitaciones del análisis
</h3>


<div class="limitacion">


<p>
La ventana reciente contiene únicamente
<strong>136 observaciones</strong>.

Por esta razón, las pruebas de cambio de distribución tienen
menor capacidad para detectar diferencias pequeñas y deben
interpretarse como evidencia de monitoreo y no como prueba
definitiva de estabilidad.
</p>


<p>
La variable temporal disponible es
<code>fecha_vinculacion</code>.

Esta variable representa la fecha de incorporación del distribuidor
y no una fecha de scoring, transacción o actualización de comportamiento.

Por esta razón, el análisis temporal permite comparar el perfil
de los distribuidores vinculados recientemente frente al histórico,
pero no medir directamente cambios en el comportamiento de
clientes existentes.
</p>


<p>
Adicionalmente, la partición original utilizada para entrenar y
evaluar el modelo SOLARIS fue realizada mediante un
<code>train_test_split</code>
<strong>aleatorio y no temporal</strong>.
</p>


<p>
En consecuencia, algunos registros pertenecientes a la ventana
reciente pudieron haber formado parte del conjunto utilizado
durante el desarrollo original del modelo.

Por esta razón, el análisis de distributional shift debe
interpretarse como una
<strong>
comparación temporal exploratoria del dataset
</strong>
y no como una validación
<strong>out-of-time completamente independiente</strong>.
</p>


<p>
Finalmente, el dataset disponible no contiene atributos protegidos
explícitos como género o etnia.

La evaluación de fairness se limita, por tanto, a las dimensiones
disponibles de:

<strong>
rango de edad, nivel educativo y ciudad.
</strong>
</p>


<p>
Los resultados no permiten afirmar que el modelo sea justo
respecto a características que no están presentes en los datos.
</p>


</div>


</div>

</body>

</html>
"""


# ============================================================
# 6. GUARDAR REPORTE FINAL
# ============================================================

REPORTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)


REPORT_FINAL_PATH = (
    REPORTS_PATH
    / "reporte_validacion_riesgos_SOLARIS.html"
)


with open(
    REPORT_FINAL_PATH,
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        html_reporte
    )


# ============================================================
# 7. CONFIRMACIÓN
# ============================================================

print(
    "Reporte final generado correctamente."
)

print(
    "Ruta:",
    REPORT_FINAL_PATH
)

print(
    "\nResumen del reporte:"
)

print(
    "- KS: sin alertas significativas"
)

print(
    "- PSI: no requerido"
)

print(
    "- AUC por subgrupos: sin brechas > 0.05"
)

print(
    "- Demographic Parity: sin brechas > 0.10"
)

print(
    "- Equalized Odds: riesgos detectados en TPR"
)

print(
    "- Riesgo principal: ciudad, brecha TPR = 0.2533"
)

print(
    "- Reporte listo para revisión por pares"
)

Reporte final generado correctamente.
Ruta: c:\Users\anaso\Downloads\proyecto-solaris\reports\reporte_validacion_riesgos_SOLARIS.html

Resumen del reporte:
- KS: sin alertas significativas
- PSI: no requerido
- AUC por subgrupos: sin brechas > 0.05
- Demographic Parity: sin brechas > 0.10
- Equalized Odds: riesgos detectados en TPR
- Riesgo principal: ciudad, brecha TPR = 0.2533
- Reporte listo para revisión por pares
